In [1]:
import numpy as np
import pandas as pd 
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotnine as pm
%matplotlib inline
import statsmodels.formula.api as smf
import statsmodels.api as sm
import scipy.stats as stats
import warnings
from sklearn.linear_model import LinearRegression
warnings.filterwarnings('ignore')
import re


In [2]:
intake_data = pd.read_csv('/kaggle/input/datasets/micahluftig/cleaned-data/intake_data.csv')
outcome_data = pd.read_csv('/kaggle/input/datasets/micahluftig/cleaned-data/outcome_data.csv')

In [3]:
import sqlite3
import pandas as pd

# ==========================================
# 1. DATABASE SETUP LAYER
# ==========================================
def setup_database(conn: sqlite3.Connection, intake_df: pd.DataFrame, outcome_df: pd.DataFrame):
    """
    Seeds the staging tables and constructs indexing for optimization.
    """
    try:
        intake_df.to_sql('intake', conn, index=False, if_exists='replace')
        outcome_df.to_sql('outcome', conn, index=False, if_exists='replace')
        print('Database tables created successfully.')
    except ValueError:
        print('Tables already exist. Skipping creation step.')

    # Build lookup indices for fast query execution
    conn.execute("CREATE INDEX IF NOT EXISTS idx_animal ON intake(animal_id);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_outcome_animal_datetime ON outcome(animal_id, datetime_out);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_intake_animal_datetime ON intake(animal_id, datetime_in);")
    conn.commit()
    

# ==========================================
# 2. EXTRACTION LAYER (Your SQL logic)
# ==========================================
def extract_appointment_data(conn: sqlite3.Connection) -> pd.DataFrame:
    """Extracts chronological intake-to-discharge episodes."""
    query = """
    WITH matches AS (
        SELECT
            i.apt_id AS intake_id,
            i.animal_id,
            i.datetime_in AS intake_time,
            o.datetime_out AS discharge_time,
            o.outcome_category,
            o.outcome_subcategory,
            ROW_NUMBER() OVER (PARTITION BY i.apt_id ORDER BY o.datetime_out ASC) AS rn
        FROM intake i
        JOIN outcome o
            ON o.animal_id = i.animal_id
           AND o.datetime_out > i.datetime_in
    )
    SELECT
        m.intake_id AS intake_id,
        m.animal_id,
        m.intake_time,
        m.discharge_time,
        i.intake_type,
        i.intake_reason,
        m.outcome_category,
        m.outcome_subcategory,
        ROUND(julianday(m.discharge_time) - julianday(i.datetime_in), 2) AS los_days
    FROM matches m
    LEFT JOIN intake i ON i.apt_id = m.intake_id
    WHERE m.rn = 1
    ORDER BY m.intake_time;
    """
    return pd.read_sql_query(query, conn)

def extract_patient_data(conn: sqlite3.Connection) -> pd.DataFrame:
    """Extracts a collapsed master list unique to each animal patient."""
    query = """
    WITH LatestIntake AS (
        SELECT 
            animal_id,
            age_in_years AS age_at_first_visit,
            spp,
            akc_group,
            pregnant_o_nursing,
            datetime_in
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY animal_id
                       ORDER BY datetime_in DESC
                   ) AS rn
            FROM intake
        ) i_sub
        WHERE rn = 1
    ),
    MatchedOutcomes AS (
        SELECT 
            i.animal_id,
            i.age_at_first_visit,
            i.spp,
            i.akc_group,
            i.pregnant_o_nursing,
            o.datetime_out AS outcome_time,
            o.sex_out,
            ROW_NUMBER() OVER (
                PARTITION BY i.animal_id
                ORDER BY o.datetime_out ASC
            ) AS outcome_rn
        FROM LatestIntake i
        LEFT JOIN outcome o
            ON i.animal_id = o.animal_id
           AND o.datetime_out >= i.datetime_in
    )
    SELECT 
        animal_id,
        age_at_first_visit,
        spp,
        akc_group,
        outcome_time,
        sex_out,
        pregnant_o_nursing
    FROM MatchedOutcomes
    WHERE outcome_rn = 1
    ORDER BY animal_id;
    """
    return pd.read_sql_query(query, conn)

# ==========================================
# 3. TRANSFORMATION LAYER (Object -> Datetime64)
# ==========================================
def clean_extracted_dataframe(df: pd.DataFrame, date_cols: list) -> pd.DataFrame:
    """Handles object-to-datetime type-casting seamlessly inside the pipeline."""
    df = df.copy()
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    return df

In [4]:
def p_data_line(intake_df: pd.DataFrame, conn: sqlite3.Connection, outcome_df: pd.DataFrame) -> pd.DataFrame:
    """Sets up the database, extracts patient records, and cleans datetimes."""
    # Step 1: Run database setup normally
    setup_pipeline_database(conn, intake_df, outcome_df)
    
    # Step 2: Extract data using the database connection
    extracted_df = extract_patient_data(conn)
    
    # Step 3: Pass the dataframe into the cleaning pipe
    date_columns = ['outcome_time']
    return extracted_df.pipe(clean_extracted_dataframe, date_cols=date_columns)


def apt_data_line(intake_df: pd.DataFrame, conn: sqlite3.Connection, outcome_df: pd.DataFrame) -> pd.DataFrame:
    """Sets up the database, extracts appointment records, and cleans datetimes."""
    # Step 1: Run database setup normally
    setup_pipeline_database(conn, intake_df, outcome_df)
    
    # Step 2: Extract data using the database connection
    extracted_df = extract_appointment_data(conn)
    
    # Step 3: Pass the dataframe into the cleaning pipe
    date_columns = ['intake_time', 'discharge_time']
    return extracted_df.pipe(clean_extracted_dataframe, date_cols=date_columns)


In [5]:
import sqlite3

# Establish the database connection
conn = sqlite3.connect("shelter_pipeline.db")

# Generate both dataframes successfully
p_data = p_data_line(intake_data, conn, outcome_data)
apt_data = apt_data_line(intake_data, conn, outcome_data)

# Close the connection securely
conn.close()


NameError: name 'setup_pipeline_database' is not defined

In [ ]:
print(p_data.head())

In [ ]:
print(apt_data.head())

In [ ]:
def table_check(df, df_name, unique_id) -> pd.DataFrame:
    print(f'Beginning errorcheck for: {df_name} ***')
    print(f'\n--- Null Value Check ---')
    for column in df.columns: 
        null_count = df[column].isna().sum() 
        print(f'{column} nulls: {null_count}')
    print('\n')
    
    print(f'--- Integrity Check ---')
    if unique_id in df.columns:
        duplicate_count = df[unique_id].duplicated().sum()
        print(f'Total {unique_id} duplicates: {duplicate_count}')
    else:
        print(f"WARNING: {unique_id} not found for duplication check.")
    print('\n')
    print(f'\n--- Data Type Audit (DF.dtypes) ---')
    print(df.dtypes)
    
    print(f'\n--- Numerical Sanity Check (DF.describe) ---')
    
    numeric_cols = df.select_dtypes(include = ['int64', 'float64', 'Int64', 'int32']).columns
    if not numeric_cols.empty:
        print(df[numeric_cols].describe())
    else:
        print('No standard numerical columns found for description.')

    print(f'\n--- Categorical Value Audit (Top 10 Counts) ---')
    object_cols = df.select_dtypes(include=['object', 'string[python]']).columns

    for column in object_cols:
        print(f'\n{column.upper()}:')
        print(df[column].value_counts().nlargest(10))

    

    print('\n\n\n')

    return df

In [ ]:
x = table_check(p_data, 'p_data', 'animal_id')


In [6]:
!pip freeze > requirements.txt